### Etapa 1: Ingestão e Blindagem de Dados

Nesta célula, nosso "garçom" (biblioteca `requests`) vai à internet buscar o formato JSON contendo o histórico do Dólar dos últimos 15 dias.

**Proteção Cloud:** Como rodaremos este script na nuvem, a AwesomeAPI pode bloquear nosso acesso achando que somos um ataque automatizado. Para evitar isso, instruímos a IA a enviar um "User-Agent", disfarçando nosso robô como um navegador Google Chrome comum.

**🤖 PROMPT ENVIADO PARA A IA:**
> "Crie um script em Python que acesse a AwesomeAPI para buscar a cotação do dólar dos últimos 15 dias em formato JSON. Muito importante: passe um cabeçalho (header) de 'User-Agent' simulando o navegador Google Chrome para evitar bloqueios de segurança (erro 403) no Google Colab. Percorra os dados, extraia a data convertendo o 'timestamp' numérico para o formato Ano-Mês-Dia e extraia o valor. Guarde essas duas informações blindadas dentro de uma Tupla e adicione em uma lista chamada historico_dolar."

In [ ]:
import requests
from datetime import datetime

# URL da API pública AwesomeAPI
url_api = "https://economia.awesomeapi.com.br/json/daily/USD-BRL/15"

# Camuflagem: Finge ser o navegador Google Chrome para evitar bloqueio no Colab
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Connection": "keep-alive"
}

resposta = requests.get(url_api, headers=headers)

dados_json = resposta.json()

historico_dolar = []

for dia in dados_json:
    # Extraímos a data via 'timestamp' universal.
    ts_inteiro = int(dia["timestamp"])
    data = datetime.fromtimestamp(ts_inteiro).strftime("%Y-%m-%d")
    valor = float(dia["bid"])
    
    # Empacotando em Tupla (imutável)
    tupla_diaria = (data, valor)
    historico_dolar.append(tupla_diaria)

print("Coleta concluída! Últimos dias carregados:")
for registro in historico_dolar:
    print(f'{registro}')

### Etapa 2: Transformação de Dados via Lookup Table

Para evitar um código espaguete cheio de `if/elif/else`, utilizamos uma Matriz (Listas dentro de Listas). Cruzando o Eixo Y (Volatilidade) com o Eixo X (Tendência), a aplicação acessa a decisão de compra de forma instantânea através dos índices!

**🤖 PROMPT ENVIADO PARA A IA:**
> "Refatore a lógica de decisão de compra cambial baseada na variação do dia. Não utilize múltiplos blocos if/elif/else para a decisão final. Crie uma Matriz Multidimensional (Lookup Table) onde a Linha representa a Volatilidade (Baixa/Alta) e a Coluna representa a Tendência (Queda/Estável/Alta). Calcule os índices matematicamente e consulte a matriz diretamente para obter o status estratégico de cada dia."

In [ ]:
# Eixo Y (Linhas 0, 1) | Eixo X (Colunas 0, 1, 2)
matriz_decisao = [
    #       0             1                2
    ["✅ Comprar", "⏸️ Manter", "⏳ Aguardar Queda"],       # 0 Vol. Baixa
    ["⚠️ Risco: Comprar", "⚠️ Risco: Manter", "🛑 Paralisar"] # 1 Vol. Alta
]

relatorio_analitico = []

# Lendo do dia mais antigo para o mais novo
for i in range(len(historico_dolar)-2, -1, -1):
    hoje = historico_dolar[i][1]
    ontem = historico_dolar[i+1][1]
    variacao = hoje - ontem
    
    # Convertendo cenário em índices (Matemática pura)
    tendencia_idx = 0 if variacao < -0.02 else (2 if variacao > 0.02 else 1)
    vol_idx = 1 if abs(variacao) > 0.06 else 0
    
    # Consulta instantânea O(1)
    decisao = matriz_decisao[vol_idx][tendencia_idx]
    
    relatorio_analitico.append({
        "Data": historico_dolar[i][0],
        "Cotacao_USD": hoje,
        "Variacao_Dia": round(variacao, 4),
        "Acao_Estrategica": decisao
    })

print("Lógica de Matriz O(1) processada com sucesso!")
for registro in relatorio_analitico:
    print(f'{registro}') 

### Etapa 3: Visualização de Dados (Data Viz)

Como estamos em um arquivo iterativo `.ipynb`, podemos renderizar imagens nativamente! Utilizamos o `matplotlib` para plotar um gráfico de linhas dinâmico, evidenciando a tendência da cotação.

**🤖 PROMPT ENVIADO PARA A IA:**
> "Crie uma célula de código importando a biblioteca matplotlib. Extraia as datas e os valores do nosso dicionário relatorio_analitico e construa um gráfico de linhas (lineplot). Formate o visual de forma profissional: adicione grid tracejado, rotacione o eixo X em 45 graus para facilitar a leitura das datas, pinte a linha do dólar de verde e garanta que a prancheta seja renderizada nativamente abaixo da célula."

In [ ]:
import matplotlib.pyplot as plt

# Preparando vetores para o gráfico
datas = [item["Data"] for item in relatorio_analitico]
valores_usd = [item["Cotacao_USD"] for item in relatorio_analitico]

# Configurando a prancheta (Figure)
fig, eixo1 = plt.subplots(figsize=(10, 5))

eixo1.plot(datas, valores_usd, color="green", marker="o", linewidth=2, label="Dólar (BRL)")
eixo1.set_xlabel("Timeline", fontweight="bold")
eixo1.set_ylabel("Preço USD", color="green", fontweight="bold")
eixo1.tick_params(axis="x", rotation=45)

plt.title("Monitoramento Cambial - TechSolutions", fontsize=14, pad=15)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()

# Exibe o gráfico nativamente no Jupyter
plt.show()

### Etapa 4: Geração de Artefato e UX no Excel

A diretoria consome relatórios. O Pandas assume a liderança transformando nossos dicionários em um `DataFrame`. Após salvar o arquivo, entraremos com a biblioteca `openpyxl` para formatar nativamente a planilha.

**Magia Cloud:** Como o Jupyter está rodando em um servidor do Google, nós implementamos um bloco de "UX Automática" no final que força o navegador a fazer o download da planilha pronta diretamente para a máquina do usuário final.

**🤖 PROMPT ENVIADO PARA A IA:**
> "Crie a etapa final chamando o Pandas para converter a base (relatorio_analitico) em DataFrame. Mostre um preview com display() e exporte para Excel (relatorio_auditoria_cambial.xlsx) sem index. Após isso, utilize o módulo openpyxl para estilizar a planilha recém-criada: formate o intervalo como uma Tabela (Table), centralize os dados e aplique tamanhos de fonte (16 no cabeçalho, 14 nos dados). Por fim, aplique wb.close() para evitar arquivos corrompidos na nuvem e crie um bloco try/except importando google.colab.files para forçar o download automático da planilha para o usuário."

In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

# 1. Transformando num DataFrame tabular do Pandas
df = pd.DataFrame(relatorio_analitico)

print("--- PREVIEW DO RELATÓRIO EXECUTIVO ---")
display(df.head())

# 2. Exportação Bruta inicial para .xlsx
nome_arquivo = "relatorio_auditoria_cambial.xlsx"
df.to_excel(nome_arquivo, index=False)

# 3. Engenharia de UX: Estilizando nativamente com OpenPyXL
wb = load_workbook(nome_arquivo)
ws = wb.active

fonte_cabecalho = Font(bold=True, size=16)
fonte_dados = Font(size=14)
alinhamento = Alignment(horizontal="center", vertical="center")

# Varrendo células para aplicar alinhamento e fontes
for linha in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
    for celula in linha:
        celula.alignment = alinhamento
        celula.font = fonte_cabecalho if celula.row == 1 else fonte_dados

# Expandindo a largura das colunas
for col in ws.columns:
    ws.column_dimensions[col[0].column_letter].width = 25

# Aplicando componente de Tabela Interativa (Zebrada)
ref_tabela = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
tabela = Table(displayName="TabelaCambio", ref=ref_tabela)
estilo = TableStyleInfo(name="TableStyleMedium9", showRowStripes=True)
tabela.tableStyleInfo = estilo
ws.add_table(tabela)

# 4. Fechamento de Memória Crucial para Ambientes Linux/Colab!
wb.save(nome_arquivo)
wb.close()

print(f"\n✅ SUCESSO! Planilha {nome_arquivo} gerada e finalizada!")

# 5. UX Automática: Se estiver no Google Colab, baixa o arquivo na hora!
try:
    from google.colab import files
    files.download(nome_arquivo)
except ImportError:
    pass # Rodando no VS Code local, ignora e segue a vida normal

--- PREVIEW DO RELATÓRIO EXECUTIVO ---


,Data,Cotacao_USD,Variacao_Dia,Acao_Estrategica
0,2026-07-16,5.11390,0.0231,⏳ Aguardar Queda
1,2026-07-17,5.10470,-0.0092,⏸️ Manter
2,2026-07-19,5.12070,0.0160,⏸️ Manter
3,2026-07-20,5.10510,-0.0156,⏸️ Manter
4,2026-07-21,5.08770,-0.0174,⏸️ Manter
5,2026-07-22,5.06537,-0.0223,✅ Comprar
6,2026-07-23,5.09700,0.0316,⏳ Aguardar Queda
7,2026-07-24,5.07850,-0.0185,⏸️ Manter
8,2026-07-26,5.08250,0.0040,⏸️ Manter
9,2026-07-27,5.12860,0.0461,⏳ Aguardar Queda



✅ SUCESSO! Planilha relatorio_auditoria_cambial.xlsx gerada e finalizada!
